In [33]:
from pathlib import Path
import os
import sys
from pyspark.sql import SparkSession




# ======================================================
# Spark Session
# ======================================================

spark = (
    SparkSession.builder
    .appName("Conexao PostgreSQL")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "/Users/eduardoalberto/LoadFile/output")
    .config("spark.jars", "/Users/eduardoalberto/opt/spark-3.5.4/jars/postgresql-42.7.5.jar")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print(f"Spark Version : {spark.version}")

# ======================================================
# PostgreSQL
# ======================================================

CONF = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": int(os.getenv("POSTGRES_PORT", "5432")),
    "database": os.getenv("POSTGRES_DATABASE", "data_lake"),
    "schema": "bronze",
    "user": os.getenv("POSTGRES_USER", "postgres"),
    "password": os.getenv("POSTGRES_PASSWORD", "postgre123"),
    "driver": "org.postgresql.Driver",
    "batchsize": 5000,
    "fetchsize": 1000,
    "numPartitions": 4,
}

print(f"Conectando ao PostgreSQL em {CONF['host']}:{CONF['port']} / {CONF['database']} com usuário {CONF['user']}")

STAGING = "/Users/eduardoalberto/LoadFile/staging"
PROCESSED = "/Users/eduardoalberto/LoadFile/processed"


Spark Version : 3.5.4
Conectando ao PostgreSQL em localhost:5432 / data_lake com usuário postgres


In [3]:
! ls -l /Users/eduardoalberto/LoadFile/staging/



total 0
drwxr-xr-x@ 38 eduardoalberto  staff  1216 Aug  3 20:57 csv
drwxr-xr-x@ 14 eduardoalberto  staff   448 Aug  3 21:05 kmz
drwxr-xr-x@  2 eduardoalberto  staff    64 Aug  3 20:57 ods


In [ ]:
# ======================================================
# Bulk Load CSV para PostgreSQL
# ======================================================

import re
from pathlib import Path
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.types import StringType

CSV_DIR = "/Users/eduardoalberto/LoadFile/staging/csv"


def sanitize_table_name(file_name):
    """Converte o nome do arquivo em um nome de tabela válido no PostgreSQL."""
    table_name = re.sub(r"[^a-zA-Z0-9_]+", "_", file_name).strip("_").lower()
    return table_name[:63]


def clean_null_bytes(dataframe):
    """Remove o byte nulo, que não é aceito pelo PostgreSQL em texto UTF-8."""
    string_columns = {
        field.name
        for field in dataframe.schema.fields
        if isinstance(field.dataType, StringType)
    }

    return dataframe.select(
        *[
            regexp_replace(col(column_name), "\\u0000", "").alias(column_name)
            if column_name in string_columns
            else col(column_name)
            for column_name in dataframe.columns
        ]
    )


def bulk_load_csv(csv_path, postgres_config):
    table_name = sanitize_table_name(csv_path.stem)
    qualified_table_name = f"{postgres_config['schema']}.{table_name}"
    print(f"Processando {csv_path.name} -> {qualified_table_name}")

    dataframe = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("encoding", "ISO-8859-1")
        .option("escape", '"')
        .csv(str(csv_path))
    )
    dataframe = clean_null_bytes(dataframe)

    jdbc_url = (
        f"jdbc:postgresql://{postgres_config['host']}:{postgres_config['port']}"
        f"/{postgres_config['database']}"
    )

    (
        dataframe.write
        .format("jdbc")
        .mode("overwrite")
        .option("url", jdbc_url)
        .option("dbtable", qualified_table_name)
        .option("user", postgres_config["user"])
        .option("password", postgres_config["password"])
        .option("driver", postgres_config["driver"])
        .option("batchsize", postgres_config["batchsize"])
        .option("numPartitions", postgres_config["numPartitions"])
        .save()
    )

    return {"arquivo": csv_path.name, "tabela": qualified_table_name, "sucesso": True}


csv_files = sorted(Path(CSV_DIR).glob("*.csv"))
results = []

for csv_file in csv_files:
    try:
        results.append(bulk_load_csv(csv_file, POSTGRES))
    except Exception as error:
        table_name = sanitize_table_name(csv_file.stem)
        qualified_table_name = f"{POSTGRES['schema']}.{table_name}"
        print(f"Erro ao processar {csv_file.name}: {error}")
        results.append({"arquivo": csv_file.name, "tabela": qualified_table_name, "sucesso": False})

print("\nResumo do bulk load:")
for result in results:
    status = "Sucesso" if result["sucesso"] else "Erro"
    print(f"{status}: {result['arquivo']} -> {result['tabela']}")

sucessos = sum(result["sucesso"] for result in results)
print(f"Total: {sucessos}/{len(results)} arquivo(s) carregado(s) com sucesso")

### consulta tabelas postgres


In [34]:
jdbc_url = (
    f"jdbc:postgresql://{CONF['host']}:{CONF['port']}/{CONF['database']}"
)

t1 = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT * FROM bronze.dicionario_base_de_dados_crai_a_partir_de_2025")
    .option("user", CONF["user"])
    .option("password", CONF["password"])
    .option("driver", CONF["driver"])
    .load()
)

t2 = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT * FROM bronze.basecadunicoimigrantesmsp082020")
    .option("user", CONF["user"])
    .option("password", CONF["password"])
    .option("driver", CONF["driver"])
    .load()
)



t3 = (    
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT * FROM bronze.bancocrai2014a2024_sistematizacao_geoinfo_atualizada")
    .option("user", CONF["user"])
    .option("password", CONF["password"])
    .option("driver", CONF["driver"])
    .load()  
)

t4 = (    
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT * FROM bronze.dicionario_base_de_dados_crai_a_partir_de_2025")
    .option("user", CONF["user"])
    .option("password", CONF["password"])
    .option("driver", CONF["driver"])
    .load()  
)

t5 = (    
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT * FROM bronze.dicionario_de_bancocrai2014a2019")
    .option("user", CONF["user"])
    .option("password", CONF["password"])
    .option("driver", CONF["driver"])
    .load()  
)

t6 = (    
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT * FROM bronze.dicionariobasecadunicoimigrantesmsp082020_1")
    .option("user", CONF["user"])
    .option("password", CONF["password"])
    .option("driver", CONF["driver"])
    .load()  
)

t7 = (    
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT * FROM bronze.programabolsafamilia_202501")
    .option("user", CONF["user"])
    .option("password", CONF["password"])
    .option("driver", CONF["driver"])
    .load()  
)


t7.show(10, truncate=False)

+------------------------------------------------------------------------------------------+
|conteudo                                                                                  |
+------------------------------------------------------------------------------------------+
|Arquivo KMZ processado: /Users/eduardoalberto/LoadFile/raw/programabolsafamilia_202501.kmz|
+------------------------------------------------------------------------------------------+



In [ ]:
from pyspark.sql.functions import split, col, trim

if not {"nome_campo", "descricao_campo"}.issubset(t1.columns):
    coluna_origem = t4.columns[0]
    colunas_originais = [c for c in t1.columns if c != coluna_origem]

    t4 = (
        t4.withColumn("_split", split(col(coluna_origem), ";", 2))
        .select(
            *colunas_originais,
            trim(col("_split")[0]).alias("nome_campo"),
            trim(col("_split")[1]).alias("descricao_campo")
        )
    )

t4.show(truncate=False)

+----------------------------+--------------------------------------------------------------------------------------------------+
|nome_campo                  |descricao_campo                                                                                   |
+----------------------------+--------------------------------------------------------------------------------------------------+
|dh_atendimento              |Data de realiza��o do atendimento                                                                 |
|Equipamento                 |Local/unidade onde o atendimento aconteceu                                                        |
|Tipo de Atendimento         |Categoria do servi�o prestado.                                                                    |
|pessoas                     |Quantidade de pessoas atendidas no registro                                                       |
|Pa�s de Nascimento          |Pa�s onde a pessoa nasceu.                                  

In [ ]:
from pyspark.sql.functions import split, col, trim

if not {"nome_campo", "descricao_campo"}.issubset(t1.columns):
    coluna_origem = t1.columns[0]
    colunas_originais = [c for c in t1.columns if c != coluna_origem]

    t1 = (
        t1.withColumn("_split", split(col(coluna_origem), ";", 2))
        .select(
            *colunas_originais,
            trim(col("_split")[0]).alias("nome_campo"),
            trim(col("_split")[1]).alias("descricao_campo")
        )
    )

t1.show(truncate=False)

In [ ]:
from pyspark.sql.functions import split, col

coluna = "COD_SEXO_PESSOA;ANO_NASCIMENTO;IDADE_EM_2020;COD_RACA_COR_PESSO"

t2 = t2.withColumn(
    "_split",
    split(col(coluna), ";")
).select(
    *[c for c in t2.columns if c != coluna],
    col("_split")[0].cast("string").alias("COD_SEXO_PESSOA"),
    col("_split")[1].cast("int").alias("ANO_NASCIMENTO"),
    col("_split")[2].cast("int").alias("IDADE_EM_2020"),
    col("_split")[3].cast("string").alias("COD_RACA_COR_PESSO")
)

t2.show(truncate=False)

In [14]:
from pyspark.sql.functions import col, element_at, first, length, split, trim

coluna_origem = t5.columns[0]

linhas_dicionario = (
    t5.select(col(coluna_origem).alias("linha"))
    .unionByName(spark.createDataFrame([(coluna_origem,)], ["linha"]))
)

t5_estrutura = (
    linhas_dicionario.select(
        trim(element_at(split(col("linha"), ";", 2), 1)).alias("nome_campo"),
        trim(element_at(split(col("linha"), ";", 2), 2)).alias("descricao_campo")
    )
    .filter(
        (length(col("nome_campo")) > 0)
        & (length(col("descricao_campo")) > 0)
    )
)

t5_pivotado = (
    t5_estrutura
    .groupBy()
    .pivot("nome_campo")
    .agg(first("descricao_campo"))
)

t5_pivotado.show(truncate=False)

+------------------------------------+--------------------+--------------------------+------------------------------------+-------------------------+----------------------------------------------------------+-----------------------------------------------------------------------------------------------+-------------------+-------------------+------------------------+------------+--------------+--------------------------+----+-------------------+
|cidade_ingresso                     |condicoes_moradia   |cor_raca                  |data_cadastro                       |data_entrada_brasil      |demanda_1                                                 |dif_dtcad_dtingr_meses                                                                         |distrito_moradia   |escolaridade       |fonte_renda             |fx_etaria   |pais_origem   |sabendo_crai              |sexo|situacao_migratoria|
+------------------------------------+--------------------+--------------------------+--------------

In [32]:
from pyspark.sql.functions import col, split, trim

if {"nome_campo", "descricao_campo"}.issubset(t6.columns):
    t6 = t6.select("nome_campo", "descricao_campo")
elif len(t6.columns) >= 2:
    t6 = t6.select(
        trim(col(t6.columns[0])).alias("nome_campo"),
        trim(col(t6.columns[1])).alias("descricao_campo")
    )
else:
    coluna_origem = t6.columns[0]
    t6 = t6.select(
        trim(split(col(coluna_origem), ";", 2)[0]).alias("nome_campo"),
        trim(split(col(coluna_origem), ";", 2)[1]).alias("descricao_campo")
    )

t6.show(truncate=False)

+---------------------------+-----------------------------------------------------------+
|nome_campo                 |descricao_campo                                            |
+---------------------------+-----------------------------------------------------------+
|DICIONÁRIO DE VARIÁVEIS    |NULL                                                       |
|VARIÁVEL                   |DESCRIÇÃO                                                  |
|COD_SEXO_PESSOA            |Sexo: 1 - Masculino, 2 - Feminino                          |
|ANO_NASCIMENTO             |Ano de nascimento da pessoa                                |
|IDADE_EM_2020              |Idade completa em 2020                                     |
|COD_RACA_COR_PESSOA        |Cor ou raça: campo numérico de 1 posição variando de 1 a 5,|
|1-Branca                   |NULL                                                       |
|2-Preta                    |NULL                                                       |
|3-Amarela

In [ ]:
# ======================================================
# Carga dos dataframes para a camada Silver
# ======================================================

from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.types import StringType

SILVER_SCHEMA = "silver"


def clean_for_silver(dataframe):
    """Remove bytes nulos e padroniza os nomes das colunas para a camada Silver."""
    dataframe = dataframe.select(
        *[
            regexp_replace(col(column_name), "\\u0000", "").alias(column_name)
            if isinstance(dataframe.schema[column_name].dataType, StringType)
            else col(column_name)
            for column_name in dataframe.columns
        ]
    )

    return dataframe


def load_to_silver(dataframe, table_name):
    """Grava um dataframe na tabela Silver correspondente."""
    qualified_table_name = f"{SILVER_SCHEMA}.{table_name}"
    print(f"Carregando {qualified_table_name}...")

    (
        clean_for_silver(dataframe)
        .write
        .format("jdbc")
        .mode("overwrite")
        .option("url", jdbc_url)
        .option("dbtable", qualified_table_name)
        .option("user", CONF["user"])
        .option("password", CONF["password"])
        .option("driver", CONF["driver"])
        .option("batchsize", CONF["batchsize"])
        .option("numPartitions", CONF["numPartitions"])
        .save()
    )

    print(f"Concluído: {qualified_table_name}")


silver_tables = {
    "t1": (t1, "dicionario_crai_2025_t1"),
    "t2": (t2, "base_cadunico_imigrantes_2020_t2"),
    "t3": (t3, "base_crai_2014_2024_t3"),
    "t4": (t4, "dicionario_crai_2025_t4"),
    "t5": (t5, "dicionario_crai_2014_2019_t5"),
    "t6": (t6, "dicionario_cadunico_imigrantes_2020_t6"),
    "t7": (t7, "programa_bolsa_familia_2025_t7"),
}

silver_results = []

for dataframe_name, (dataframe, table_name) in silver_tables.items():
    try:
        load_to_silver(dataframe, table_name)
        silver_results.append({
            "dataframe": dataframe_name,
            "tabela": f"{SILVER_SCHEMA}.{table_name}",
            "sucesso": True,
        })
    except Exception as error:
        print(f"Erro ao carregar {dataframe_name}: {error}")
        silver_results.append({
            "dataframe": dataframe_name,
            "tabela": f"{SILVER_SCHEMA}.{table_name}",
            "sucesso": False,
            "erro": str(error),
        })

print("\nResumo da carga Silver:")
for result in silver_results:
    status = "Sucesso" if result["sucesso"] else "Erro"
    print(f"{status}: {result['dataframe']} -> {result['tabela']}")

Carregando silver.dicionario_crai_2025_t1...
Concluído: silver.dicionario_crai_2025_t1
Carregando silver.base_cadunico_imigrantes_2020_t2...
Concluído: silver.base_cadunico_imigrantes_2020_t2
Carregando silver.base_crai_2014_2024_t3...


Concluído: silver.base_crai_2014_2024_t3
Carregando silver.dicionario_crai_2025_t4...
Concluído: silver.dicionario_crai_2025_t4
Carregando silver.dicionario_crai_2014_2019_t5...
Concluído: silver.dicionario_crai_2014_2019_t5
Carregando silver.dicionario_cadunico_imigrantes_2020_t6...
Concluído: silver.dicionario_cadunico_imigrantes_2020_t6
Carregando silver.programa_bolsa_familia_2025_t7...
Concluído: silver.programa_bolsa_familia_2025_t7

Resumo da carga Silver:
Sucesso: t1 -> silver.dicionario_crai_2025_t1
Sucesso: t2 -> silver.base_cadunico_imigrantes_2020_t2
Sucesso: t3 -> silver.base_crai_2014_2024_t3
Sucesso: t4 -> silver.dicionario_crai_2025_t4
Sucesso: t5 -> silver.dicionario_crai_2014_2019_t5
Sucesso: t6 -> silver.dicionario_cadunico_imigrantes_2020_t6
Sucesso: t7 -> silver.programa_bolsa_familia_2025_t7


26/09/23 14:39:54 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE